# CineContextHGT

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from torch import nn
from torch_geometric.data import HeteroData
from torch_geometric.nn import HGTConv, Linear

from common.eval import build_user_item_dict, evaluate_ranking


/Users/alexandro/DataspellProjects/recommend-system/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class CineContextHGT(nn.Module):

    def __init__(
            self,
            num_nodes_dict: dict,
            metadata,
            hidden_channels: int = 128,
            out_channels: int = 128,
            num_heads: int = 4,
            num_layers: int = 2,
            dropout: float = 0.2,
    ):
        super().__init__()


        """
        Initialize core components of the heterogeneous graph transformer model.

        - embeddings: Separate embedding layers for each node type in the graph.
        - convs: List of HGT convolution layers for message passing.
        - residuals: Residual connections to stabilize deep transformations.
        - norms: Normalization layers applied after each convolution.

        """
        self.embeddings = nn.ModuleDict({
            node_type: nn.Embedding(num_nodes, hidden_channels)
            for node_type, num_nodes in num_nodes_dict.items()
        })
        self.convs = nn.ModuleList()
        self.residuals = nn.ModuleList()
        self.norms = nn.ModuleList()

        for _ in range(num_layers):
            self.convs.append(
                HGTConv(
                    in_channels=hidden_channels,
                    out_channels=hidden_channels,
                    metadata=metadata,
                    heads=num_heads,
                )
            )
            self.residuals.append(nn.Linear(hidden_channels, hidden_channels))
            self.norms.append(nn.LayerNorm(hidden_channels))

        self.dropout = nn.Dropout(dropout)
        self.user_projection = Linear(hidden_channels, out_channels)
        self.movie_projection = Linear(hidden_channels, out_channels)


    """
    Encodes all node types in the heterogeneous graph into latent representations.
    """
    def encode(self, data: HeteroData) -> dict:
        x_dict = {
            node_type: self.embeddings[node_type](data[node_type].node_id)
            for node_type in data.node_types
        }

        for conv, residual, norm in zip(self.convs, self.residuals, self.norms):
            h_dict = conv(x_dict, data.edge_index_dict)

            updated = {}
            for node_type, x in x_dict.items():
                h = h_dict.get(node_type, x)
                h = h + residual(x)
                h = norm(h)
                h = F.relu(h)
                h = self.dropout(h)
                updated[node_type] = h

            x_dict = updated

        x_dict["user"] = self.user_projection(x_dict["user"])
        x_dict["movie"] = self.movie_projection(x_dict["movie"])
        return x_dict


    """
    Forward pass of the model.
    """
    def forward(self, data: HeteroData) -> tuple[torch.Tensor, torch.Tensor]:
        encoded = self.encode(data)
        return encoded["user"], encoded["movie"]

    def score_pairs(
            self,
            user_idx: torch.Tensor,
            item_idx: torch.Tensor,
            data: HeteroData,
    ) -> torch.Tensor:
        user_out, movie_out = self.forward(data)
        return (user_out[user_idx] * movie_out[item_idx]).sum(dim=-1)


    """
    Computes the full user–item relevance score matrix.
    """
    def full_score_matrix(self, data: HeteroData) -> torch.Tensor:
        user_out, movie_out = self.forward(data)
        return user_out @ movie_out.T

In [3]:
"""
Compute Bayesian Personalized Ranking loss.
"""
def bpr_loss(pos_scores: torch.Tensor, neg_scores: torch.Tensor) -> torch.Tensor:
    return -F.logsigmoid(pos_scores - neg_scores).mean()


"""
Yield mini-batches of positive user-item interactions.
"""
def iterate_minibatches(
    pos_u: torch.Tensor,
    pos_i: torch.Tensor,
    batch_size: int,
    shuffle: bool = True,
):
    n = pos_u.size(0)
    indices = torch.arange(n, device=pos_u.device)

    if shuffle:
        indices = indices[torch.randperm(n, device=pos_u.device)]

    for start in range(0, n, batch_size):
        batch_idx = indices[start : start + batch_size]
        yield pos_u[batch_idx], pos_i[batch_idx]


"""
Sample one negative item per user in the batch.
"""
def sample_negative_items_for_batch(
    batch_u: torch.Tensor,
    train_user_items: dict,
    item2idx: dict,
    idx2user: dict,
    device: torch.device,
) -> torch.Tensor:
    all_item_ids = np.array(list(item2idx.keys()))
    neg_item_ids = []

    for u_idx in batch_u.detach().cpu().numpy():
        user_id = idx2user[int(u_idx)]
        seen_items = train_user_items.get(user_id, set())

        while True:
            item_id = int(np.random.choice(all_item_ids))
            if item_id not in seen_items:
                neg_item_ids.append(item2idx[item_id])
                break

    return torch.tensor(neg_item_ids, dtype=torch.long, device=device)


"""
Train the model for one epoch using BPR optimization.
"""
def train_one_epoch(
    model: CineContextHGT,
    data: HeteroData,
    train_df: pd.DataFrame,
    item2idx: dict,
    idx2user: dict,
    pos_u: torch.Tensor,
    pos_i: torch.Tensor,
    optimizer: torch.optim.Optimizer,
    batch_size: int = 4096,
) -> float:
    model.train()

    train_user_items = build_user_item_dict(
        train_df,
        user_col="user_id",
        item_col="movie_id",
    )

    total_loss = 0.0
    total_examples = 0

    for batch_u, batch_pos_i in iterate_minibatches(
        pos_u=pos_u,
        pos_i=pos_i,
        batch_size=batch_size,
        shuffle=True,
    ):
        optimizer.zero_grad()

        batch_neg_i = sample_negative_items_for_batch(
            batch_u=batch_u,
            train_user_items=train_user_items,
            item2idx=item2idx,
            idx2user=idx2user,
            device=batch_u.device,
        )

        user_out, movie_out = model(data)
        pos_scores = (user_out[batch_u] * movie_out[batch_pos_i]).sum(dim=-1)
        neg_scores = (user_out[batch_u] * movie_out[batch_neg_i]).sum(dim=-1)

        loss = bpr_loss(pos_scores, neg_scores)
        loss.backward()
        optimizer.step()

        batch_size_actual = batch_u.size(0)
        total_loss += loss.item() * batch_size_actual
        total_examples += batch_size_actual

    return total_loss / max(total_examples, 1)


"""
Generate top-k movie recommendations for evaluation users.
"""
def generate_topk_recommendations(
    model: CineContextHGT,
    data: HeteroData,
    train_df: pd.DataFrame,
    test_df_eval: pd.DataFrame,
    user2idx: dict,
    item2idx: dict,
    idx2item: dict,
    k: int = 20,
) -> dict:
    model.eval()

    with torch.no_grad():
        score_matrix = model.full_score_matrix(data).detach().cpu().numpy()

    train_user_items = build_user_item_dict(
        train_df,
        user_col="user_id",
        item_col="movie_id",
    )

    eval_users = sorted(test_df_eval["user_id"].unique())
    recommendations = {}

    for user_id in eval_users:
        if user_id not in user2idx:
            continue

        user_scores = score_matrix[user2idx[user_id]].copy()
        seen_items = train_user_items.get(user_id, set())

        for item_id in seen_items:
            if item_id in item2idx:
                user_scores[item2idx[item_id]] = -1e9

        top_k = min(k, len(user_scores))
        top_idx = np.argpartition(-user_scores, top_k - 1)[:top_k]
        top_idx = top_idx[np.argsort(-user_scores[top_idx])]
        recommendations[user_id] = [idx2item[i] for i in top_idx]

    return recommendations


"""
Evaluate the recommender at top-k.
"""
def evaluate_model_at_k(
    model: CineContextHGT,
    data: HeteroData,
    train_df: pd.DataFrame,
    test_df_eval: pd.DataFrame,
    user2idx: dict,
    item2idx: dict,
    idx2item: dict,
    k: int,
) -> dict:
    recommendations = generate_topk_recommendations(
        model=model,
        data=data,
        train_df=train_df,
        test_df_eval=test_df_eval,
        user2idx=user2idx,
        item2idx=item2idx,
        idx2item=idx2item,
        k=k,
    )

    gt_eval = (
        test_df_eval.groupby("user_id")["movie_id"]
        .apply(list)
        .to_dict()
    )

    metrics = evaluate_ranking(
        recommendations=recommendations,
        ground_truth=gt_eval,
        k=k,
    )
    metrics["n_users_eval"] = len(gt_eval)
    return metrics

In [4]:
from dataclasses import dataclass
from typing import Dict, Tuple, List

EdgeType = Tuple[str, str, str]

@dataclass
class FeatureConfig:
    """
    Describes how a feature node type is connected in the graph.
    """
    node_type: str
    edge_type: EdgeType
    reverse_edge_type: EdgeType | None = None
    display_name: str = ""


@dataclass
class GraphSchema:
    """
    Defines semantic structure of the heterogeneous graph.
    """
    node_features: Dict[str, Dict[str, FeatureConfig]]

    def get_features(self, node_type: str) -> Dict[str, FeatureConfig]:
        return self.node_features.get(node_type, {})

@dataclass
class EdgeGroupImportance:
    edge_type: EdgeType
    base_score: float
    masked_score: float
    importance: float

@dataclass
class LocalFeatureEvidence:
    feature_name: str
    feature_node_idx: int
    support_movies: List[int]
    support_count: int

@dataclass
class LocalExplanation:
    user_idx: int
    movie_idx: int
    base_score: float
    edge_group_importance: List[EdgeGroupImportance]
    local_feature_evidence: Dict[str, List[LocalFeatureEvidence]]

In [5]:
from typing import Set, Dict


class LocalSubgraphBuilder:
    """
    Extracts local neighborhood around (user, movie).
    """

    def __init__(
        self,
        data: HeteroData,
        schema: GraphSchema,
    ):
        self.data = data
        self.schema = schema

    def get_neighbors(
        self,
        node_type: str,
        node_idx: int,
        edge_type: EdgeType,
    ) -> Set[int]:
        """
        Get neighbors of a node via given edge type.
        """
        if edge_type not in self.data.edge_types:
            return set()

        edge_index = self.data[edge_type].edge_index
        src_type, _, dst_type = edge_type

        neighbors = set()

        if node_type == src_type:
            mask = edge_index[0] == node_idx
            neighbors = set(edge_index[1][mask].tolist())

        elif node_type == dst_type:
            mask = edge_index[1] == node_idx
            neighbors = set(edge_index[0][mask].tolist())

        return neighbors

    def build_context(
        self,
        user_idx: int,
        movie_idx: int,
        train_user_items: Dict[int, Set[int]],
    ) -> Dict:
        """
        Build local explanation context using schema.
        """
        watched = train_user_items.get(user_idx, set())

        context = {
            "user": user_idx,
            "movie": movie_idx,
            "features": {}
        }

        movie_features = self.schema.get_features("movie")

        for feature_name, cfg in movie_features.items():
            feature_nodes = self.get_neighbors(
                node_type="movie",
                node_idx=movie_idx,
                edge_type=cfg.edge_type,
            )

            support = {}

            for f in feature_nodes:
                support_movies = []

                for wm in watched:
                    wm_features = self.get_neighbors(
                        "movie",
                        wm,
                        cfg.edge_type
                    )
                    if f in wm_features:
                        support_movies.append(wm)

                if support_movies:
                    support[f] = support_movies

            context["features"][feature_name] = support

        return context

In [6]:
import torch
import copy
from typing import Dict, List, Optional
from torch_geometric.data import HeteroData


class RecommendationSubgraphAnalyzer:
    """
    Core explainability engine for HGT-based recommendations.

    Works in three steps:
    1. Compute base score
    2. Perturb graph (mask edges)
    3. Measure score drop (importance)
    """

    def __init__(
            self,
            model,
            data: HeteroData,
            schema: GraphSchema,
            train_user_items_idx: Dict[int, Set[int]],
            idx2item: Dict[int, int],
            idx2feature_maps: Optional[Dict[str, Dict[int, int]]] = None,
            device: Optional[torch.device] = None,
    ):
        self.model = model
        self.data = data
        self.schema = schema
        self.train_user_items_idx = train_user_items_idx
        self.idx2item = idx2item
        self.idx2feature_maps = idx2feature_maps or {}

        if device is None:
            try:
                device = next(model.parameters()).device
            except StopIteration:
                device = torch.device("cpu")

        self.device = device
        self.model.eval()

    @torch.no_grad()
    def score_pair(self, user_idx: int, movie_idx: int, data=None) -> float:
        """
        Compute score(u, m) = dot(user_embedding, movie_embedding)
        """
        graph = data if data is not None else self.data
        encoded = self.model.encode(graph)

        u = encoded["user"][user_idx]
        m = encoded["movie"][movie_idx]

        return float((u * m).sum().item())

    def clone_graph(self, data=None) -> HeteroData:
        """
        Deep copy graph for safe perturbations.
        """
        graph = data if data is not None else self.data
        return copy.deepcopy(graph)

    def mask_edge_type(self, edge_type: EdgeType, data=None) -> HeteroData:
        """
        Remove all edges of a given type.
        """
        graph = self.clone_graph(data)

        if edge_type not in graph.edge_types:
            return graph

        edge_index = graph[edge_type].edge_index
        graph[edge_type].edge_index = edge_index.new_empty((2, 0))

        return graph


    @torch.no_grad()
    def compute_edge_type_importance(self, user_idx: int, movie_idx: int, edge_type: EdgeType) -> EdgeGroupImportance:
        base_score = self.score_pair(user_idx, movie_idx, self.data)
        masked_data = self.mask_edge_type(edge_type, self.data)
        masked_score = self.score_pair(user_idx, movie_idx, masked_data)
        return EdgeGroupImportance(
            edge_type=edge_type,
            base_score=base_score,
            masked_score=masked_score,
            importance=base_score - masked_score,
        )


    def _get_neighbors(self, node_type: str, node_idx: int, edge_type: EdgeType) -> Set[int]:
        if edge_type not in self.data.edge_types:
            return set()

        edge_index = self.data[edge_type].edge_index
        src_type, _, dst_type = edge_type

        if node_type == src_type:
            mask = edge_index[0] == node_idx
            return set(edge_index[1][mask].detach().cpu().tolist())

        if node_type == dst_type:
            mask = edge_index[1] == node_idx
            return set(edge_index[0][mask].detach().cpu().tolist())

        return set()


    def build_local_feature_evidence(self, user_idx: int, movie_idx: int) -> Dict[str, List[LocalFeatureEvidence]]:
        watched_movies = self.train_user_items_idx.get(user_idx, set())
        result: Dict[str, List[LocalFeatureEvidence]] = {}

        movie_features = self.schema.get_features("movie")

        for feature_name, cfg in movie_features.items():
            feature_nodes = self._get_neighbors("movie", movie_idx, cfg.edge_type)
            evidences: List[LocalFeatureEvidence] = []

            for feature_node_idx in feature_nodes:
                support_movies = []
                for watched_movie_idx in watched_movies:
                    watched_movie_features = self._get_neighbors("movie", watched_movie_idx, cfg.edge_type)
                    if feature_node_idx in watched_movie_features:
                        support_movies.append(watched_movie_idx)

                if support_movies:
                    evidences.append(
                        LocalFeatureEvidence(
                            feature_name=feature_name,
                            feature_node_idx=feature_node_idx,
                            support_movies=sorted(support_movies),
                            support_count=len(support_movies),
                        )
                    )

            evidences.sort(key=lambda x: x.support_count, reverse=True)
            result[feature_name] = evidences

        return result


    def explain(
            self,
            user_idx: int,
            movie_idx: int,
            edge_types: Optional[List[EdgeType]] = None,
    ) -> LocalExplanation:
        if edge_types is None:
            edge_types = list(self.data.edge_types)

        base_score = self.score_pair(user_idx, movie_idx, self.data)
        edge_group_importance = [
            self.compute_edge_type_importance(user_idx, movie_idx, edge_type)
            for edge_type in edge_types
        ]
        edge_group_importance.sort(key=lambda x: x.importance, reverse=True)

        local_feature_evidence = self.build_local_feature_evidence(user_idx, movie_idx)

        return LocalExplanation(
            user_idx=user_idx,
            movie_idx=movie_idx,
            base_score=base_score,
            edge_group_importance=edge_group_importance,
            local_feature_evidence=local_feature_evidence,
        )


    def pretty_print(self, explanation: LocalExplanation, top_k_features: int = 5):
        print(f"User idx: {explanation.user_idx}")
        print(f"Movie idx: {explanation.movie_idx} -> raw movie id: {self.idx2item.get(explanation.movie_idx)}")
        print(f"Base score: {explanation.base_score:.4f}")
        print("\n[Edge-group importance]")
        for row in explanation.edge_group_importance:
            print(f"{row.edge_type}: Δscore = {row.importance:.4f} (masked={row.masked_score:.4f})")

        print("\n[Local feature evidence]")
        for feature_name, evidences in explanation.local_feature_evidence.items():
            print(f"\n{feature_name}:")
            if not evidences:
                print("  no supporting evidence in watched history")
                continue
            feature_map = self.idx2feature_maps.get(feature_name, {})
            for ev in evidences[:top_k_features]:
                raw_feature_id = feature_map.get(ev.feature_node_idx, ev.feature_node_idx)
                raw_support_movies = [self.idx2item.get(i, i) for i in ev.support_movies]
                print(
                    f"  feature_idx={ev.feature_node_idx} raw_id={raw_feature_id} | "
                    f"support_count={ev.support_count} | support_movies={raw_support_movies}"
                )

In [7]:
class ExplanationFormatter:

    """
    Converts structured explanation into readable format.
    """
    def format(self, context: Dict, top_k: int = 3) -> Dict:
        result = {}

        for feature, values in context["features"].items():
            ranked = sorted(values.items(), key=lambda x: len(x[1]), reverse=True)
            result[feature] = ranked[:top_k]

        return result

In [8]:
rates = pd.read_csv("../user_movie_rates2.csv")
users = pd.read_csv("../users.csv")
movies = pd.read_csv("../movies.csv")

movie_genres = pd.read_csv("../movie_genres.csv")
genres = pd.read_csv("../genres.csv")

movie_actors = pd.read_csv("../movie_actors.csv")
actors = pd.read_csv("../actors.csv")

movie_directors = pd.read_csv("../movie_directors.csv")
directors = pd.read_csv("../directors.csv")

movie_countries = pd.read_csv("../movie_countries.csv")
countries = pd.read_csv("../countries.csv")

movie_tags = pd.read_csv("../movie_tag.csv")
tags = pd.read_csv("../tags.csv")

age_groups = pd.read_csv("../age_groups.csv")
occupations = pd.read_csv("../occupations.csv")

print("rates:", rates.shape, rates.columns.tolist())
print("tags:", tags.shape, tags.columns.tolist())
print("movies:", movies.shape, movies.columns.tolist())
print("movie_genres:", movie_genres.shape, movie_genres.columns.tolist())
print("movie_directors:", movie_directors.shape, movie_directors.columns.tolist())
print("movie_actors:", movie_actors.shape, movie_actors.columns.tolist())
print("movie_countries:", movie_countries.shape, movie_countries.columns.tolist())
print("movie_tags:", movie_tags.shape, movie_tags.columns.tolist())
print("user_ages:", age_groups.shape, age_groups.columns.tolist())
print("user_occ:", occupations.shape, occupations.columns.tolist())

rates: (151000, 4) ['user_id', 'movie_id', 'rating', 'datetime']
tags: (48075, 2) ['tag_id', 'tag_name']
movies: (3433, 7) ['movie_id', 'title', 'year', 'rated', 'plot', 'imdb_rating', 'imdb_id']
movie_genres: (6408, 2) ['movie_id', 'genre_id']
movie_directors: (3676, 2) ['movie_id', 'director_id']
movie_actors: (10285, 2) ['movie_id', 'actor_id']
movie_countries: (4765, 2) ['movie_id', 'country_id']
movie_tags: (450912, 2) ['movie_id', 'tag_id']
user_ages: (7, 2) ['group_id', 'group_label']
user_occ: (21, 2) ['occ_id', 'occ_name']


In [9]:
import random

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


Параметры эксперимента

In [10]:
context_order = [
    "genres_directors_actors_countries_demographics"
]

threshold_values = [5.0]
min_pos_values = [5]
k_values = [20]

EPOCHS = 100
TEST_RATIO = 0.2

HIDDEN_CHANNELS = 128
OUT_CHANNELS = 128
NUM_HEADS = 4
NUM_LAYERS = 2
DROPOUT = 0.2

LR = 5e-4
WEIGHT_DECAY = 1e-5
BATCH_SIZE = 4096

TOP_N_DIRECTORS = 200
TOP_N_ACTORS = 200

In [11]:
def build_entity_index_map(df: pd.DataFrame, entity_col: str):
    entity_ids = sorted(df[entity_col].dropna().astype(int).unique())
    entity2idx = {entity_id: idx for idx, entity_id in enumerate(entity_ids)}
    idx2entity = {idx: entity_id for entity_id, idx in entity2idx.items()}
    return entity2idx, idx2entity

def build_string_index_map(df: pd.DataFrame, col: str):
    values = sorted(df[col].dropna().astype(str).unique())
    value2idx = {value: idx for idx, value in enumerate(values)}
    idx2value = {idx: value for value, idx in value2idx.items()}
    return value2idx, idx2value


def keep_top_n_entities_by_frequency(
    relation_df: pd.DataFrame,
    entity_col: str,
    top_n: int,
):
    top_entities = (
        relation_df[entity_col]
        .value_counts()
        .head(top_n)
        .index
    )
    return relation_df[relation_df[entity_col].isin(top_entities)].copy()


def filter_context_by_train_items(
    context_df: pd.DataFrame,
    train_item_ids: set,
    movie_col: str = "movie_id",
):
    return context_df[context_df[movie_col].isin(train_item_ids)].copy()

In [12]:
def _stage_feature_set(stage: str):
    if stage is None:
        return set()
    stage = str(stage).strip().lower()
    if stage == "" or stage == "raw":
        return set()
    return set(part for part in stage.split("_") if part)


def build_hgt_heterodata(
    stage: str,
    train_df: pd.DataFrame,
    user2idx: dict,
    item2idx: dict,
    movie_genres_train: pd.DataFrame = None,
    genre2idx: dict = None,
    movie_directors_train: pd.DataFrame = None,
    director2idx: dict = None,
    movie_actors_train: pd.DataFrame = None,
    actor2idx: dict = None,
    movie_countries_train: pd.DataFrame = None,
    country2idx: dict = None,
    users_train: pd.DataFrame = None,
    gender2idx: dict = None,
    occupation2idx: dict = None,
    age_group2idx: dict = None,
):
    feature_set = _stage_feature_set(stage)
    data = HeteroData()

    data["user"].node_id = torch.arange(len(user2idx), dtype=torch.long)
    data["movie"].node_id = torch.arange(len(item2idx), dtype=torch.long)

    user_idx = train_df["user_id"].map(user2idx).to_numpy()
    movie_idx = train_df["movie_id"].map(item2idx).to_numpy()

    data["user", "interacts", "movie"].edge_index = torch.tensor(
        np.vstack([user_idx, movie_idx]),
        dtype=torch.long
    )
    data["movie", "rev_interacts", "user"].edge_index = torch.tensor(
        np.vstack([movie_idx, user_idx]),
        dtype=torch.long
    )

    if "genres" in feature_set:
        data["genre"].node_id = torch.arange(len(genre2idx), dtype=torch.long)

        tmp = movie_genres_train.copy()
        tmp["movie_idx"] = tmp["movie_id"].map(item2idx)
        tmp["genre_idx"] = tmp["genre_id"].map(genre2idx)
        tmp = tmp.dropna(subset=["movie_idx", "genre_idx"])

        movie_idx = tmp["movie_idx"].astype(int).to_numpy()
        genre_idx = tmp["genre_idx"].astype(int).to_numpy()

        data["movie", "has_genre", "genre"].edge_index = torch.tensor(
            np.vstack([movie_idx, genre_idx]),
            dtype=torch.long
        )
        data["genre", "rev_has_genre", "movie"].edge_index = torch.tensor(
            np.vstack([genre_idx, movie_idx]),
            dtype=torch.long
        )

    if "directors" in feature_set:
        data["director"].node_id = torch.arange(len(director2idx), dtype=torch.long)

        tmp = movie_directors_train.copy()
        tmp["movie_idx"] = tmp["movie_id"].map(item2idx)
        tmp["director_idx"] = tmp["director_id"].map(director2idx)
        tmp = tmp.dropna(subset=["movie_idx", "director_idx"])

        movie_idx = tmp["movie_idx"].astype(int).to_numpy()
        director_idx = tmp["director_idx"].astype(int).to_numpy()

        data["movie", "has_director", "director"].edge_index = torch.tensor(
            np.vstack([movie_idx, director_idx]),
            dtype=torch.long
        )
        data["director", "rev_has_director", "movie"].edge_index = torch.tensor(
            np.vstack([director_idx, movie_idx]),
            dtype=torch.long
        )

    if "actors" in feature_set:
        data["actor"].node_id = torch.arange(len(actor2idx), dtype=torch.long)

        tmp = movie_actors_train.copy()
        tmp["movie_idx"] = tmp["movie_id"].map(item2idx)
        tmp["actor_idx"] = tmp["actor_id"].map(actor2idx)
        tmp = tmp.dropna(subset=["movie_idx", "actor_idx"])

        movie_idx = tmp["movie_idx"].astype(int).to_numpy()
        actor_idx = tmp["actor_idx"].astype(int).to_numpy()

        data["movie", "has_actor", "actor"].edge_index = torch.tensor(
            np.vstack([movie_idx, actor_idx]),
            dtype=torch.long
        )
        data["actor", "rev_has_actor", "movie"].edge_index = torch.tensor(
            np.vstack([actor_idx, movie_idx]),
            dtype=torch.long
        )

    if "countries" in feature_set:
        data["country"].node_id = torch.arange(len(country2idx), dtype=torch.long)

        tmp = movie_countries_train.copy()
        tmp["movie_idx"] = tmp["movie_id"].map(item2idx)
        tmp["country_idx"] = tmp["country_id"].map(country2idx)
        tmp = tmp.dropna(subset=["movie_idx", "country_idx"])

        movie_idx = tmp["movie_idx"].astype(int).to_numpy()
        country_idx = tmp["country_idx"].astype(int).to_numpy()

        data["movie", "has_country", "country"].edge_index = torch.tensor(
            np.vstack([movie_idx, country_idx]),
            dtype=torch.long
        )
        data["country", "rev_has_country", "movie"].edge_index = torch.tensor(
            np.vstack([country_idx, movie_idx]),
            dtype=torch.long
        )

    if "demographics" in feature_set:
        data["gender"].node_id = torch.arange(len(gender2idx), dtype=torch.long)
        data["occupation"].node_id = torch.arange(len(occupation2idx), dtype=torch.long)
        data["age_group"].node_id = torch.arange(len(age_group2idx), dtype=torch.long)

        tmp = users_train.copy()
        tmp["user_idx"] = tmp["user_id"].map(user2idx)
        tmp["gender_idx"] = tmp["gender"].astype(str).map(gender2idx)
        tmp["occupation_idx"] = tmp["occupation"].astype(str).map(occupation2idx)
        tmp["age_group_idx"] = tmp["age_group_id"].map(age_group2idx)

        ug = tmp.dropna(subset=["user_idx", "gender_idx"]).copy()
        u = ug["user_idx"].astype(int).to_numpy()
        g = ug["gender_idx"].astype(int).to_numpy()
        data["user", "has_gender", "gender"].edge_index = torch.tensor(np.vstack([u, g]), dtype=torch.long)
        data["gender", "rev_has_gender", "user"].edge_index = torch.tensor(np.vstack([g, u]), dtype=torch.long)

        uo = tmp.dropna(subset=["user_idx", "occupation_idx"]).copy()
        u = uo["user_idx"].astype(int).to_numpy()
        o = uo["occupation_idx"].astype(int).to_numpy()
        data["user", "has_occupation", "occupation"].edge_index = torch.tensor(np.vstack([u, o]), dtype=torch.long)
        data["occupation", "rev_has_occupation", "user"].edge_index = torch.tensor(np.vstack([o, u]), dtype=torch.long)

        ua = tmp.dropna(subset=["user_idx", "age_group_idx"]).copy()
        u = ua["user_idx"].astype(int).to_numpy()
        a = ua["age_group_idx"].astype(int).to_numpy()
        data["user", "has_age_group", "age_group"].edge_index = torch.tensor(np.vstack([u, a]), dtype=torch.long)
        data["age_group", "rev_has_age_group", "user"].edge_index = torch.tensor(np.vstack([a, u]), dtype=torch.long)

    return data


def build_num_nodes_dict(
    stage: str,
    user2idx: dict,
    item2idx: dict,
    genre2idx: dict = None,
    director2idx: dict = None,
    actor2idx: dict = None,
    country2idx: dict = None,
    gender2idx: dict = None,
    occupation2idx: dict = None,
    age_group2idx: dict = None,
):
    feature_set = _stage_feature_set(stage)
    num_nodes_dict = {
        "user": len(user2idx),
        "movie": len(item2idx),
    }

    if "genres" in feature_set:
        num_nodes_dict["genre"] = len(genre2idx)

    if "directors" in feature_set:
        num_nodes_dict["director"] = len(director2idx)

    if "actors" in feature_set:
        num_nodes_dict["actor"] = len(actor2idx)

    if "countries" in feature_set:
        num_nodes_dict["country"] = len(country2idx)

    if "demographics" in feature_set:
        num_nodes_dict["gender"] = len(gender2idx)
        num_nodes_dict["occupation"] = len(occupation2idx)
        num_nodes_dict["age_group"] = len(age_group2idx)

    return num_nodes_dict


In [13]:
from types import SimpleNamespace

def make_synthetic_cold_start_split(
    interactions: pd.DataFrame,
    user_col: str = "user_id",
    item_col: str = "movie_id",
    time_col: str = "datetime",
    cold_user_fraction: float = 0.2,
    cold_n: int = 3,
    min_interactions_for_cold: int = 20,
    random_state: int = 42,
):
    """
    Synthetic cold start:
    - для части пользователей (cold_users) в train оставляем только первые cold_n взаимодействий
    - остальные их взаимодействия переносим в test
    - для остальных пользователей оставляем последнее взаимодействие в test
    """
    df = interactions.copy()
    df = df.sort_values([user_col, time_col]).reset_index(drop=True)

    user_counts = df.groupby(user_col).size()
    eligible_cold_users = user_counts[user_counts >= min_interactions_for_cold].index.to_numpy()

    rng = np.random.default_rng(random_state)
    n_cold_users = int(round(len(eligible_cold_users) * cold_user_fraction))
    if n_cold_users == 0 and cold_user_fraction > 0 and len(eligible_cold_users) > 0:
        n_cold_users = 1

    cold_users = set(
        rng.choice(eligible_cold_users, size=n_cold_users, replace=False).tolist()
    ) if n_cold_users > 0 else set()

    train_parts = []
    test_parts = []
    split_rows = []

    for user_id, user_df in df.groupby(user_col, sort=False):
        user_df = user_df.sort_values(time_col)

        if user_id in cold_users:
            train_user = user_df.iloc[:cold_n].copy()
            test_user = user_df.iloc[cold_n:].copy()

            if len(test_user) == 0:
                train_user = user_df.iloc[:-1].copy()
                test_user = user_df.iloc[-1:].copy()

            split_type = "cold"
        else:
            if len(user_df) <= 1:
                train_user = user_df.copy()
                test_user = user_df.iloc[0:0].copy()
            else:
                train_user = user_df.iloc[:-1].copy()
                test_user = user_df.iloc[-1:].copy()

            split_type = "warm"

        train_parts.append(train_user)
        if len(test_user) > 0:
            test_parts.append(test_user)

        split_rows.append({
            user_col: user_id,
            "split_type": split_type,
            "original_interactions": len(user_df),
            "train_interactions": len(train_user),
            "test_interactions": len(test_user),
        })

    train_df = pd.concat(train_parts, ignore_index=True) if train_parts else df.iloc[0:0].copy()
    test_df = pd.concat(test_parts, ignore_index=True) if test_parts else df.iloc[0:0].copy()
    split_stats = pd.DataFrame(split_rows)

    return SimpleNamespace(
        train_df=train_df,
        test_df=test_df,
        cold_users=cold_users,
        eligible_cold_users=set(eligible_cold_users.tolist()),
        split_stats=split_stats,
    )


def summarize_split_result(split_result):
    split_stats = split_result.split_stats
    rows = []

    for group_name, mask in [
        ("cold_users", split_stats["split_type"] == "cold"),
        ("warm_users", split_stats["split_type"] == "warm"),
    ]:
        group_df = split_stats.loc[mask]
        rows.append({
            "group": group_name,
            "n_users": len(group_df),
            "mean_train_interactions": round(group_df["train_interactions"].mean(), 2) if len(group_df) else 0.0,
            "mean_test_interactions": round(group_df["test_interactions"].mean(), 2) if len(group_df) else 0.0,
        })

    rows.append({
        "group": "all_users",
        "n_users": split_stats["user_id"].nunique() if len(split_stats) else 0,
        "mean_train_interactions": round(split_stats["train_interactions"].mean(), 2) if len(split_stats) else 0.0,
        "mean_test_interactions": round(split_stats["test_interactions"].mean(), 2) if len(split_stats) else 0.0,
    })

    return pd.DataFrame(rows)

In [14]:
from common.data_prep import build_edges, filter_users_min_pos
from common.split import temporal_train_test_split
from common.indexing import build_index_maps

def fit_single_hgt_cold_start_experiment_with_artifacts(
    stage: str,
    threshold: float,
    min_pos: int,
    rates: pd.DataFrame,
    users: pd.DataFrame,
    movie_genres: pd.DataFrame,
    movie_directors: pd.DataFrame,
    movie_actors: pd.DataFrame,
    movie_countries: pd.DataFrame,
    cold_user_fraction: float = 0.2,
    cold_n: int = 3,
    min_interactions_for_cold: int = 20,
    epochs: int = 30,
    hidden_channels: int = 128,
    out_channels: int = 128,
    num_heads: int = 4,
    num_layers: int = 2,
    dropout: float = 0.2,
    lr: float = 5e-4,
    weight_decay: float = 1e-5,
    batch_size: int = 4096,
    top_n_directors: int = 200,
    top_n_actors: int = 200,
    device: torch.device = torch.device("cpu"),
    random_state: int = 42,
):
    """
    Train a single CineContextHGT experiment in synthetic cold-start setup
    and return all artifacts needed for evaluation and explainability.

    Cold-start logic:
    - for a fraction of eligible users, only the first `cold_n` interactions
      are kept in train
    - the remaining interactions of these users go to test
    - for warm users, the last interaction is used for test
    """

    movie_directors_top = keep_top_n_entities_by_frequency(
        relation_df=movie_directors,
        entity_col="director_id",
        top_n=top_n_directors,
    )

    movie_actors_top = keep_top_n_entities_by_frequency(
        relation_df=movie_actors,
        entity_col="actor_id",
        top_n=top_n_actors,
    )

    df_pos = build_edges(
        data=rates,
        threshold=threshold,
        user_col="user_id",
        item_col="movie_id",
        rating_col="rating",
        time_col="datetime",
    )

    df_pos = filter_users_min_pos(
        data=df_pos,
        min_pos=min_pos,
        user_col="user_id",
    )

    if len(df_pos) == 0:
        raise ValueError("No positive interactions left after threshold/min_pos filtering.")

    split_result = make_synthetic_cold_start_split(
        interactions=df_pos,
        user_col="user_id",
        item_col="movie_id",
        time_col="datetime",
        cold_user_fraction=cold_user_fraction,
        cold_n=cold_n,
        min_interactions_for_cold=min_interactions_for_cold,
        random_state=random_state,
    )

    train_df = split_result.train_df
    test_df = split_result.test_df

    if len(train_df) == 0:
        raise ValueError("Cold-start split produced empty train_df.")
    if len(test_df) == 0:
        raise ValueError("Cold-start split produced empty test_df.")
    if len(split_result.cold_users) == 0:
        raise ValueError("Cold-start split produced no cold users.")

    user2idx, idx2user, item2idx, idx2item = build_index_maps(
        train_df=train_df,
        user_col="user_id",
        item_col="movie_id",
    )

    if len(user2idx) == 0 or len(item2idx) == 0:
        raise ValueError("Empty user/item vocabulary after building index maps.")

    train_item_ids = set(train_df["movie_id"].unique())
    train_user_ids = set(train_df["user_id"].unique())

    movie_genres_train = filter_context_by_train_items(
        context_df=movie_genres,
        train_item_ids=train_item_ids,
        movie_col="movie_id",
    )
    movie_directors_train = filter_context_by_train_items(
        context_df=movie_directors_top,
        train_item_ids=train_item_ids,
        movie_col="movie_id",
    )
    movie_actors_train = filter_context_by_train_items(
        context_df=movie_actors_top,
        train_item_ids=train_item_ids,
        movie_col="movie_id",
    )
    movie_countries_train = filter_context_by_train_items(
        context_df=movie_countries,
        train_item_ids=train_item_ids,
        movie_col="movie_id",
    )

    users_train = users[users["user_id"].isin(train_user_ids)].copy()

    genre2idx, idx2genre = build_entity_index_map(movie_genres_train, "genre_id")
    director2idx, idx2director = build_entity_index_map(movie_directors_train, "director_id")
    actor2idx, idx2actor = build_entity_index_map(movie_actors_train, "actor_id")
    country2idx, idx2country = build_entity_index_map(movie_countries_train, "country_id")

    gender2idx, idx2gender = build_string_index_map(users_train, "gender")
    occupation2idx, idx2occupation = build_string_index_map(users_train, "occupation")
    age_group2idx, idx2age_group = build_entity_index_map(users_train, "age_group_id")

    # evaluate only on cold users, and only on users/items present in train vocab
    test_df_eval = test_df[
        test_df["user_id"].isin(split_result.cold_users)
        & test_df["user_id"].isin(user2idx.keys())
        & test_df["movie_id"].isin(item2idx.keys())
    ].copy()

    if len(test_df_eval) == 0:
        raise ValueError("Filtered cold-start evaluation set is empty.")

    data_stage = build_hgt_heterodata(
        stage=stage,
        train_df=train_df,
        user2idx=user2idx,
        item2idx=item2idx,
        movie_genres_train=movie_genres_train,
        genre2idx=genre2idx,
        movie_directors_train=movie_directors_train,
        director2idx=director2idx,
        movie_actors_train=movie_actors_train,
        actor2idx=actor2idx,
        movie_countries_train=movie_countries_train,
        country2idx=country2idx,
        users_train=users_train,
        gender2idx=gender2idx,
        occupation2idx=occupation2idx,
        age_group2idx=age_group2idx,
    ).to(device)

    num_nodes_dict = build_num_nodes_dict(
        stage=stage,
        user2idx=user2idx,
        item2idx=item2idx,
        genre2idx=genre2idx,
        director2idx=director2idx,
        actor2idx=actor2idx,
        country2idx=country2idx,
        gender2idx=gender2idx,
        occupation2idx=occupation2idx,
        age_group2idx=age_group2idx,
    )

    model_stage = CineContextHGT(
        num_nodes_dict=num_nodes_dict,
        metadata=data_stage.metadata(),
        hidden_channels=hidden_channels,
        out_channels=out_channels,
        num_heads=num_heads,
        num_layers=num_layers,
        dropout=dropout,
    ).to(device)

    optimizer = torch.optim.Adam(
        model_stage.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )

    pos_edge_index = data_stage["user", "interacts", "movie"].edge_index
    pos_u = pos_edge_index[0]
    pos_i = pos_edge_index[1]

    epoch_losses = []
    for epoch in range(1, epochs + 1):
        loss = train_one_epoch(
            model=model_stage,
            data=data_stage,
            train_df=train_df,
            item2idx=item2idx,
            idx2user=idx2user,
            pos_u=pos_u,
            pos_i=pos_i,
            optimizer=optimizer,
            batch_size=batch_size,
        )
        epoch_losses.append(loss)
        print(f"Epoch {epoch:02d}/{epochs} | cold_n={cold_n} | loss={loss:.4f}")

    recommendations = generate_topk_recommendations(
        model=model_stage,
        data=data_stage,
        train_df=train_df,
        test_df_eval=test_df_eval,
        user2idx=user2idx,
        item2idx=item2idx,
        idx2item=idx2item,
        k=20,
    )

    ground_truth = build_user_item_dict(
        test_df_eval,
        user_col="user_id",
        item_col="movie_id",
    )

    train_user_items_raw = build_user_item_dict(
        train_df,
        user_col="user_id",
        item_col="movie_id",
    )
    train_user_items_idx = {
        user2idx[u]: {item2idx[i] for i in items if i in item2idx}
        for u, items in train_user_items_raw.items()
        if u in user2idx
    }

    schema = GraphSchema(
        node_features={
            "movie": {
                "genre": FeatureConfig(
                    node_type="genre",
                    edge_type=("movie", "has_genre", "genre"),
                    reverse_edge_type=("genre", "rev_has_genre", "movie"),
                    display_name="Genre",
                ),
                "actor": FeatureConfig(
                    node_type="actor",
                    edge_type=("movie", "has_actor", "actor"),
                    reverse_edge_type=("actor", "rev_has_actor", "movie"),
                    display_name="Actor",
                ),
            },
            "user": {
                "gender": FeatureConfig(
                    node_type="gender",
                    edge_type=("user", "has_gender", "gender"),
                    reverse_edge_type=("gender", "rev_has_gender", "user"),
                    display_name="Gender",
                ),
                "occupation": FeatureConfig(
                    node_type="occupation",
                    edge_type=("user", "has_occupation", "occupation"),
                    reverse_edge_type=("occupation", "rev_has_occupation", "user"),
                    display_name="Occupation",
                ),
                "age_group": FeatureConfig(
                    node_type="age_group",
                    edge_type=("user", "has_age_group", "age_group"),
                    reverse_edge_type=("age_group", "rev_has_age_group", "user"),
                    display_name="Age group",
                ),
            },
        }
    )

    split_summary = summarize_split_result(split_result)

    metrics_at_5 = evaluate_ranking(recommendations=recommendations, ground_truth=ground_truth, k=5)
    metrics_at_10 = evaluate_ranking(recommendations=recommendations, ground_truth=ground_truth, k=10)
    metrics_at_20 = evaluate_ranking(recommendations=recommendations, ground_truth=ground_truth, k=20)

    artifacts = {
        "model": model_stage,
        "data": data_stage,
        "train_df": train_df,
        "test_df": test_df,
        "test_df_eval": test_df_eval,
        "recommendations": recommendations,
        "ground_truth": ground_truth,
        "schema": schema,
        "user2idx": user2idx,
        "idx2user": idx2user,
        "item2idx": item2idx,
        "idx2item": idx2item,
        "idx2feature_maps": {
            "genre": idx2genre,
            "director": idx2director,
            "actor": idx2actor,
            "country": idx2country,
            "gender": idx2gender,
            "occupation": idx2occupation,
            "age_group": idx2age_group,
        },
        "train_user_items_idx": train_user_items_idx,
        "epoch_losses": epoch_losses,
        "split_result": split_result,
        "split_summary": split_summary,
        "cold_users_raw": split_result.cold_users,
        "cold_users_idx": {user2idx[u] for u in split_result.cold_users if u in user2idx},
        "config": {
            "stage": stage,
            "threshold": threshold,
            "min_pos": min_pos,
            "cold_user_fraction": cold_user_fraction,
            "cold_n": cold_n,
            "min_interactions_for_cold": min_interactions_for_cold,
            "epochs": epochs,
            "hidden_channels": hidden_channels,
            "out_channels": out_channels,
            "num_heads": num_heads,
            "num_layers": num_layers,
            "dropout": dropout,
            "lr": lr,
            "weight_decay": weight_decay,
            "batch_size": batch_size,
            "top_n_directors": top_n_directors,
            "top_n_actors": top_n_actors,
        },
        "metrics": {
            5: metrics_at_5,
            10: metrics_at_10,
            20: metrics_at_20,
        },
    }

    return artifacts

In [57]:
cold_artifacts = fit_single_hgt_cold_start_experiment_with_artifacts(
    stage="genres_actors_countries_demographics",
    threshold=5.0,
    min_pos=5,
    rates=rates,
    users=users,
    movie_genres=movie_genres,
    movie_directors=movie_directors,
    movie_actors=movie_actors,
    movie_countries=movie_countries,
    cold_user_fraction=0.2,
    cold_n=3,
    min_interactions_for_cold=20,
    epochs=100,
    hidden_channels=128,
    out_channels=128,
    num_heads=4,
    num_layers=2,
    dropout=0.2,
    lr=5e-4,
    weight_decay=1e-5,
    batch_size=4096,
    top_n_directors=200,
    top_n_actors=200,
    device=device,
)

Epoch 01/100 | cold_n=3 | loss=0.8424
Epoch 02/100 | cold_n=3 | loss=0.4903
Epoch 03/100 | cold_n=3 | loss=0.4506
Epoch 04/100 | cold_n=3 | loss=0.4037
Epoch 05/100 | cold_n=3 | loss=0.3247
Epoch 06/100 | cold_n=3 | loss=0.2922
Epoch 07/100 | cold_n=3 | loss=0.2735
Epoch 08/100 | cold_n=3 | loss=0.2578
Epoch 09/100 | cold_n=3 | loss=0.2465
Epoch 10/100 | cold_n=3 | loss=0.2364
Epoch 11/100 | cold_n=3 | loss=0.2266
Epoch 12/100 | cold_n=3 | loss=0.2198
Epoch 13/100 | cold_n=3 | loss=0.2144
Epoch 14/100 | cold_n=3 | loss=0.2058
Epoch 15/100 | cold_n=3 | loss=0.2009
Epoch 16/100 | cold_n=3 | loss=0.1914
Epoch 17/100 | cold_n=3 | loss=0.1821
Epoch 18/100 | cold_n=3 | loss=0.1728
Epoch 19/100 | cold_n=3 | loss=0.1649
Epoch 20/100 | cold_n=3 | loss=0.1604
Epoch 21/100 | cold_n=3 | loss=0.1555
Epoch 22/100 | cold_n=3 | loss=0.1474
Epoch 23/100 | cold_n=3 | loss=0.1401
Epoch 24/100 | cold_n=3 | loss=0.1376
Epoch 25/100 | cold_n=3 | loss=0.1336
Epoch 26/100 | cold_n=3 | loss=0.1294
Epoch 27/100

In [28]:
cold_artifacts["metrics"].get(20)

{'precision': np.float64(0.18632694248234108),
 'recall': np.float64(0.1929352575558522),
 'map': np.float64(0.12541805372786705),
 'ndcg': np.float64(0.2038962766268981),
 'mrr': np.float64(0.33622977475361776),
 'hitrate': np.float64(0.7204843592330978)}

In [63]:
context_stage = [
    "raw",
    "genres",
    "genres_actors",
    "genres_actors_demographics",
    "genres_actors_demographics_directors",
    "genres_actors_demographics_directors_countries",
]

In [77]:
for stage in context_stage:
    print(f"\n🚀 Running stage: {stage}")

    res = fit_single_hgt_cold_start_experiment_with_artifacts(
        stage=stage,
        threshold=5.0,
        min_pos=5,
        rates=rates,
        users=users,
        movie_genres=movie_genres,
        movie_directors=movie_directors,
        movie_actors=movie_actors,
        movie_countries=movie_countries,
        cold_user_fraction=0.2,
        cold_n=3,
        min_interactions_for_cold=20,
        epochs=100,
        hidden_channels=128,
        out_channels=128,
        num_heads=4,
        num_layers=2,
        dropout=0.2,
        lr=5e-4,
        weight_decay=1e-5,
        batch_size=4096,
        top_n_directors=200,
        top_n_actors=200,
        device=device,
    )

    for k in (5, 10, 20):
        metrics = res["metrics"].get(k)
        results.append({
            "context_stage": stage,
            "k": k,
            "recall": metrics["recall"],
            "ndcg": metrics["ndcg"],
            "hitrate": metrics["hitrate"],
        })


🚀 Running stage: raw
Epoch 01/100 | cold_n=3 | loss=0.9480
Epoch 02/100 | cold_n=3 | loss=0.7120
Epoch 03/100 | cold_n=3 | loss=0.5270
Epoch 04/100 | cold_n=3 | loss=0.4366
Epoch 05/100 | cold_n=3 | loss=0.3993
Epoch 06/100 | cold_n=3 | loss=0.3190
Epoch 07/100 | cold_n=3 | loss=0.2914
Epoch 08/100 | cold_n=3 | loss=0.2750
Epoch 09/100 | cold_n=3 | loss=0.2642
Epoch 10/100 | cold_n=3 | loss=0.2512
Epoch 11/100 | cold_n=3 | loss=0.2447
Epoch 12/100 | cold_n=3 | loss=0.2398
Epoch 13/100 | cold_n=3 | loss=0.2346
Epoch 14/100 | cold_n=3 | loss=0.2291
Epoch 15/100 | cold_n=3 | loss=0.2224
Epoch 16/100 | cold_n=3 | loss=0.2190
Epoch 17/100 | cold_n=3 | loss=0.2115
Epoch 18/100 | cold_n=3 | loss=0.2063
Epoch 19/100 | cold_n=3 | loss=0.1995
Epoch 20/100 | cold_n=3 | loss=0.1959
Epoch 21/100 | cold_n=3 | loss=0.1913
Epoch 22/100 | cold_n=3 | loss=0.1899
Epoch 23/100 | cold_n=3 | loss=0.1834
Epoch 24/100 | cold_n=3 | loss=0.1782
Epoch 25/100 | cold_n=3 | loss=0.1744
Epoch 26/100 | cold_n=3 | lo

In [78]:
res_copy = results.copy()
res_copy

[{'context_stage': 'genres_actors_demographics',
  'k': 5,
  'recall': np.float64(0.06338164103813462),
  'ndcg': np.float64(0.24548267134802262),
  'hitrate': np.float64(0.4783047426841574)},
 {'context_stage': 'genres_actors_demographics',
  'k': 10,
  'recall': np.float64(0.11878700169790571),
  'ndcg': np.float64(0.2342649910187053),
  'hitrate': np.float64(0.6004036326942482)},
 {'context_stage': 'genres_actors_demographics',
  'k': 20,
  'recall': np.float64(0.20944301815892258),
  'ndcg': np.float64(0.22318235369172496),
  'hitrate': np.float64(0.7628657921291625)},
 {'context_stage': 'raw',
  'k': 5,
  'recall': np.float64(0.054895913209278384),
  'ndcg': np.float64(0.21484503920086434),
  'hitrate': np.float64(0.4722502522704339)},
 {'context_stage': 'raw',
  'k': 10,
  'recall': np.float64(0.10306620213897161),
  'ndcg': np.float64(0.20552394483638253),
  'hitrate': np.float64(0.6044399596367306)},
 {'context_stage': 'raw',
  'k': 20,
  'recall': np.float64(0.1896537563359393

In [21]:
analyzer = RecommendationSubgraphAnalyzer(
    model=cold_artifacts["model"],
    data=cold_artifacts["data"],
    schema=cold_artifacts["schema"],
    train_user_items_idx=cold_artifacts["train_user_items_idx"],
    idx2item=cold_artifacts["idx2item"],
    idx2feature_maps=cold_artifacts["idx2feature_maps"],
    device=device,
)

example_user_id = None
example_user_idx = None
example_movie_id = None
example_movie_idx = None
selected_explanation = None

for user_id, recs in cold_artifacts["recommendations"].items():
    if not recs:
        continue

    recommended_movie_id = recs[0]

    gt_movies = (
        cold_artifacts["test_df_eval"]
        .loc[cold_artifacts["test_df_eval"]["user_id"] == user_id, "movie_id"]
        .drop_duplicates()
        .tolist()
    )

    is_relevant = recommended_movie_id in gt_movies
    print(f"user {user_id} -> rec {recommended_movie_id} -> relevant: {is_relevant}")

    if not is_relevant:
        continue

    user_idx = cold_artifacts["user2idx"][user_id]
    movie_idx = cold_artifacts["item2idx"][recommended_movie_id]

    explanation = analyzer.explain(
        user_idx=user_idx,
        movie_idx=movie_idx,
    )

    positive_actor = False
    positive_genre = False

    for row in explanation.edge_group_importance:
        edge_type = row.edge_type
        importance = row.importance
        rel = edge_type[1]

        if "actor" in rel and importance > 0:
            positive_actor = True
        if "genre" in rel and importance > 0:
            positive_genre = True

    has_positive_context = positive_actor or positive_genre

    print(
        f"   positive_actor={positive_actor}, "
        f"positive_genre={positive_genre}, "
        f"has_positive_context={has_positive_context}"
    )

    if has_positive_context:
        example_user_id = user_id
        example_user_idx = user_idx
        example_movie_id = recommended_movie_id
        example_movie_idx = movie_idx
        selected_explanation = explanation
        break

if example_user_id is None:
    raise ValueError(
        "No relevant top-1 recommendation found with positive actor or genre importance."
    )

print("\n✅ Selected relevant context-driven case")
print("Example user_id:", example_user_id)
print("Recommended movie_id:", example_movie_id)

analyzer.pretty_print(selected_explanation, top_k_features=5)

user 31 -> rec 1198 -> relevant: False
user 37 -> rec 3114 -> relevant: False
user 44 -> rec 2762 -> relevant: True
   positive_actor=False, positive_genre=False, has_positive_context=False
user 58 -> rec 1199 -> relevant: True
   positive_actor=False, positive_genre=True, has_positive_context=True

✅ Selected relevant context-driven case
Example user_id: 58
Recommended movie_id: 1199
User idx: 53
Movie idx: 838 -> raw movie id: 1199
Base score: 3.4730

[Edge-group importance]
('movie', 'rev_interacts', 'user'): Δscore = 11.2367 (masked=-7.7636)
('user', 'interacts', 'movie'): Δscore = 9.3989 (masked=-5.9259)
('genre', 'rev_has_genre', 'movie'): Δscore = 0.0555 (masked=3.4175)
('movie', 'has_actor', 'actor'): Δscore = 0.0000 (masked=3.4730)
('user', 'has_gender', 'gender'): Δscore = 0.0000 (masked=3.4730)
('user', 'has_occupation', 'occupation'): Δscore = 0.0000 (masked=3.4730)
('user', 'has_age_group', 'age_group'): Δscore = 0.0000 (masked=3.4730)
('movie', 'has_genre', 'genre'): Δsco

In [22]:
cold_artifacts_2 = fit_single_hgt_cold_start_experiment_with_artifacts(
    stage="genres_actors_demographics",
    threshold=5.0,
    min_pos=5,
    rates=rates,
    users=users,
    movie_genres=movie_genres,
    movie_directors=movie_directors,
    movie_actors=movie_actors,
    movie_countries=movie_countries,
    cold_user_fraction=0.2,
    cold_n=1,
    min_interactions_for_cold=20,
    epochs=100,
    hidden_channels=128,
    out_channels=128,
    num_heads=4,
    num_layers=2,
    dropout=0.2,
    lr=5e-4,
    weight_decay=1e-5,
    batch_size=4096,
    top_n_directors=200,
    top_n_actors=200,
    device=device,
)

Epoch 01/100 | cold_n=1 | loss=0.6057
Epoch 02/100 | cold_n=1 | loss=0.3346
Epoch 03/100 | cold_n=1 | loss=0.3157
Epoch 04/100 | cold_n=1 | loss=0.3076
Epoch 05/100 | cold_n=1 | loss=0.3035
Epoch 06/100 | cold_n=1 | loss=0.2988
Epoch 07/100 | cold_n=1 | loss=0.2930
Epoch 08/100 | cold_n=1 | loss=0.2894
Epoch 09/100 | cold_n=1 | loss=0.2837
Epoch 10/100 | cold_n=1 | loss=0.2799
Epoch 11/100 | cold_n=1 | loss=0.2799
Epoch 12/100 | cold_n=1 | loss=0.2765
Epoch 13/100 | cold_n=1 | loss=0.2752
Epoch 14/100 | cold_n=1 | loss=0.2742
Epoch 15/100 | cold_n=1 | loss=0.2714
Epoch 16/100 | cold_n=1 | loss=0.2692
Epoch 17/100 | cold_n=1 | loss=0.2595
Epoch 18/100 | cold_n=1 | loss=0.2461
Epoch 19/100 | cold_n=1 | loss=0.2363
Epoch 20/100 | cold_n=1 | loss=0.2332
Epoch 21/100 | cold_n=1 | loss=0.2320
Epoch 22/100 | cold_n=1 | loss=0.2296
Epoch 23/100 | cold_n=1 | loss=0.2251
Epoch 24/100 | cold_n=1 | loss=0.2236
Epoch 25/100 | cold_n=1 | loss=0.2247
Epoch 26/100 | cold_n=1 | loss=0.2217
Epoch 27/100

In [24]:
cold_artifacts_2["metrics"]

{5: {'precision': np.float64(0.338719512195122),
  'recall': np.float64(0.03836269218876877),
  'map': np.float64(0.24337398373983735),
  'ndcg': np.float64(0.34532330411656914),
  'mrr': np.float64(0.5145833333333333),
  'hitrate': np.float64(0.7682926829268293)},
 10: {'precision': np.float64(0.3221036585365854),
  'recall': np.float64(0.07289679662588865),
  'map': np.float64(0.19978150406504067),
  'ndcg': np.float64(0.3319932641014067),
  'mrr': np.float64(0.5307146486643438),
  'hitrate': np.float64(0.8810975609756098)},
 20: {'precision': np.float64(0.2774390243902439),
  'recall': np.float64(0.12270204201691319),
  'map': np.float64(0.15104833399673046),
  'ndcg': np.float64(0.2974613819568659),
  'mrr': np.float64(0.5357461902507515),
  'hitrate': np.float64(0.9496951219512195)}}

In [28]:
analyzer_2 = RecommendationSubgraphAnalyzer(
    model=cold_artifacts_2["model"],
    data=cold_artifacts_2["data"],
    schema=cold_artifacts_2["schema"],
    train_user_items_idx=cold_artifacts_2["train_user_items_idx"],
    idx2item=cold_artifacts_2["idx2item"],
    idx2feature_maps=cold_artifacts_2["idx2feature_maps"],
    device=device,
)

example_user_id = None
example_user_idx = None
example_movie_id = None
example_movie_idx = None
selected_explanation = None

for user_id, recs in cold_artifacts_2["recommendations"].items():
    if not recs:
        continue

    recommended_movie_id = recs[0]

    gt_movies = (
        cold_artifacts_2["test_df_eval"]
        .loc[cold_artifacts_2["test_df_eval"]["user_id"] == user_id, "movie_id"]
        .drop_duplicates()
        .tolist()
    )

    is_relevant = recommended_movie_id in gt_movies
    print(f"user {user_id} -> rec {recommended_movie_id} -> relevant: {is_relevant}")

    if not is_relevant:
        continue

    user_idx = cold_artifacts_2["user2idx"][user_id]
    movie_idx = cold_artifacts_2["item2idx"][recommended_movie_id]

    explanation = analyzer_2.explain(
        user_idx=user_idx,
        movie_idx=movie_idx,
    )

    positive_actor = False
    positive_genre = False

    for row in explanation.edge_group_importance:
        edge_type = row.edge_type
        importance = row.importance
        rel = edge_type[1]


    has_positive_context = True

    print(
        f"   positive_actor={positive_actor}, "
        f"positive_genre={positive_genre}, "
        f"has_positive_context={has_positive_context}"
    )

    if has_positive_context:
        example_user_id = user_id
        example_user_idx = user_idx
        example_movie_id = recommended_movie_id
        example_movie_idx = movie_idx
        selected_explanation = explanation
        break

if example_user_id is None:
    raise ValueError(
        "No relevant top-1 recommendation found with positive actor or genre importance."
    )

print("\n✅ Selected relevant context-driven case")
print("Example user_id:", example_user_id)
print("Recommended movie_id:", example_movie_id)

analyzer_2.pretty_print(selected_explanation, top_k_features=5)

user 31 -> rec 1214 -> relevant: False
user 37 -> rec 2858 -> relevant: False
user 44 -> rec 2571 -> relevant: True
   positive_actor=False, positive_genre=False, has_positive_context=True

✅ Selected relevant context-driven case
Example user_id: 44
Recommended movie_id: 2571
User idx: 41
Movie idx: 1861 -> raw movie id: 2571
Base score: 3.9039

[Edge-group importance]
('user', 'interacts', 'movie'): Δscore = 9.9005 (masked=-5.9966)
('movie', 'rev_interacts', 'user'): Δscore = 6.6858 (masked=-2.7819)
('gender', 'rev_has_gender', 'user'): Δscore = 0.7776 (masked=3.1263)
('movie', 'has_genre', 'genre'): Δscore = 0.0000 (masked=3.9039)
('user', 'has_gender', 'gender'): Δscore = 0.0000 (masked=3.9039)
('user', 'has_occupation', 'occupation'): Δscore = 0.0000 (masked=3.9039)
('user', 'has_age_group', 'age_group'): Δscore = 0.0000 (masked=3.9039)
('movie', 'has_actor', 'actor'): Δscore = -0.0000 (masked=3.9039)
('actor', 'rev_has_actor', 'movie'): Δscore = -0.0694 (masked=3.9733)
('age_group

In [33]:
cold_artifacts = fit_single_hgt_cold_start_experiment_with_artifacts(
    stage="genres_actors",
    threshold=5.0,
    min_pos=5,
    rates=rates,
    users=users,
    movie_genres=movie_genres,
    movie_directors=movie_directors,
    movie_actors=movie_actors,
    movie_countries=movie_countries,
    cold_user_fraction=0.2,
    cold_n=1,
    min_interactions_for_cold=20,
    epochs=30,
    hidden_channels=128,
    out_channels=128,
    num_heads=4,
    num_layers=2,
    dropout=0.2,
    lr=5e-4,
    weight_decay=1e-5,
    batch_size=4096,
    top_n_directors=20,
    top_n_actors=20,
    device=device,
)

Epoch 01/30 | cold_n=1 | loss=0.8672
Epoch 02/30 | cold_n=1 | loss=0.4944
Epoch 03/30 | cold_n=1 | loss=0.4514
Epoch 04/30 | cold_n=1 | loss=0.4029
Epoch 05/30 | cold_n=1 | loss=0.3204
Epoch 06/30 | cold_n=1 | loss=0.2876
Epoch 07/30 | cold_n=1 | loss=0.2692
Epoch 08/30 | cold_n=1 | loss=0.2585
Epoch 09/30 | cold_n=1 | loss=0.2457
Epoch 10/30 | cold_n=1 | loss=0.2424
Epoch 11/30 | cold_n=1 | loss=0.2346
Epoch 12/30 | cold_n=1 | loss=0.2265
Epoch 13/30 | cold_n=1 | loss=0.2172
Epoch 14/30 | cold_n=1 | loss=0.2062
Epoch 15/30 | cold_n=1 | loss=0.1980
Epoch 16/30 | cold_n=1 | loss=0.1860
Epoch 17/30 | cold_n=1 | loss=0.1814
Epoch 18/30 | cold_n=1 | loss=0.1751
Epoch 19/30 | cold_n=1 | loss=0.1658
Epoch 20/30 | cold_n=1 | loss=0.1605
Epoch 21/30 | cold_n=1 | loss=0.1542
Epoch 22/30 | cold_n=1 | loss=0.1444
Epoch 23/30 | cold_n=1 | loss=0.1351
Epoch 24/30 | cold_n=1 | loss=0.1296
Epoch 25/30 | cold_n=1 | loss=0.1252
Epoch 26/30 | cold_n=1 | loss=0.1201
Epoch 27/30 | cold_n=1 | loss=0.1184
E

In [66]:
analyzer = RecommendationSubgraphAnalyzer(
    model=cold_artifacts["model"],
    data=cold_artifacts["data"],
    schema=cold_artifacts["schema"],
    train_user_items_idx=cold_artifacts["train_user_items_idx"],
    idx2item=cold_artifacts["idx2item"],
    idx2feature_maps=cold_artifacts["idx2feature_maps"],
    device=device,
)

example_user_id = None
example_user_idx = None
example_movie_id = None
example_movie_idx = None
selected_explanation = None

for user_id, recs in cold_artifacts["recommendations"].items():
    if not recs:
        continue

    recommended_movie_id = recs[0]

    gt_movies = (
        cold_artifacts["test_df_eval"]
        .loc[cold_artifacts["test_df_eval"]["user_id"] == user_id, "movie_id"]
        .drop_duplicates()
        .tolist()
    )

    is_relevant = recommended_movie_id in gt_movies
    print(f"user {user_id} -> rec {recommended_movie_id} -> relevant: {is_relevant}")

    if not is_relevant:
        continue

    user_idx = cold_artifacts["user2idx"][user_id]
    movie_idx = cold_artifacts["item2idx"][recommended_movie_id]

    explanation = analyzer.explain(
        user_idx=user_idx,
        movie_idx=movie_idx,
    )

    positive_actor = False
    positive_genre = False

    for row in explanation.edge_group_importance:
        edge_type = row.edge_type
        importance = row.importance
        rel = edge_type[1]

        if "actor" in rel and importance > 5:
            positive_actor = True

    has_positive_context = positive_actor or positive_genre

    print(
        f"   positive_actor={positive_actor}, "
        f"positive_genre={positive_genre}, "
        f"has_positive_context={has_positive_context}"
    )

    if has_positive_context:
        example_user_id = user_id
        example_user_idx = user_idx
        example_movie_id = recommended_movie_id
        example_movie_idx = movie_idx
        selected_explanation = explanation
        break

if example_user_id is None:
    raise ValueError(
        "No relevant top-1 recommendation found with positive actor or genre importance."
    )

print("\n✅ Selected relevant context-driven case")
print("Example user_id:", example_user_id)
print("Recommended movie_id:", example_movie_id)

analyzer.pretty_print(selected_explanation, top_k_features=5)

user 25 -> rec 258 -> relevant: False
user 27 -> rec 2127 -> relevant: False
user 30 -> rec 173 -> relevant: False
user 32 -> rec 3149 -> relevant: False
user 33 -> rec 1488 -> relevant: False
user 37 -> rec 173 -> relevant: False
user 41 -> rec 3699 -> relevant: False
user 48 -> rec 1894 -> relevant: False
user 65 -> rec 2395 -> relevant: False
user 66 -> rec 1488 -> relevant: False
user 68 -> rec 2040 -> relevant: False
user 80 -> rec 3149 -> relevant: False
user 86 -> rec 3519 -> relevant: False
user 88 -> rec 351 -> relevant: True
   positive_actor=False, positive_genre=False, has_positive_context=False
user 89 -> rec 2640 -> relevant: False
user 92 -> rec 3114 -> relevant: False
user 104 -> rec 2605 -> relevant: False
user 105 -> rec 3753 -> relevant: False
user 107 -> rec 2 -> relevant: False
user 108 -> rec 2640 -> relevant: False
user 123 -> rec 446 -> relevant: False
user 124 -> rec 2353 -> relevant: False
user 129 -> rec 3798 -> relevant: False
user 144 -> rec 3247 -> relevan

In [35]:
!pip install openai

  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.41.5-cp312-cp312-macosx_11_0_arm64.whl.metadata (7.3 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 15.5 MB/s  0:00:00
Using cached distro-1.9.0-py3-none-any.whl (20 kB)
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
Using cached pydantic-2.12.5-py3-none-any.whl (463 kB)
Using cached pydantic_core-2.41.5-cp312-cp312-macosx_11_0_arm64.whl (1.9 MB)
Using cached annotated_typ

In [53]:
import os
from openai import OpenAI

GROQ_URL = "https://api.groq.com/openai/v1"
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
GROQ_MODEL_NAME = "llama-3.1-8b-instant"

client = OpenAI(
    base_url=GROQ_URL,
    api_key=GROQ_API_KEY
)

In [54]:
movie_id_to_title = dict(zip(movies["movie_id"], movies["title"]))
actor_id_to_name = dict(zip(actors["actor_id"], actors["actor"]))
genre_id_to_name = dict(zip(genres["genre_id"], genres["name"]))
director_id_to_name = dict(zip(directors["director_id"], directors["director"])) if "director" in directors.columns else {}
country_id_to_name = dict(zip(countries["country_id"], countries["name"])) if "name" in countries.columns else {}
age_group_id_to_name = dict(zip(age_groups["age_group_id"], age_groups["name"])) if "name" in age_groups.columns else {}
occupation_id_to_name = dict(zip(occupations["occupation_id"], occupations["occupation"])) if "occupation" in occupations.columns else {}

In [64]:
def build_bilingual_user_prompt(payload: dict) -> str:
    return f"""
You are explaining a movie recommendation to a user.

Use ONLY the structured evidence below.

Rules:
- Do NOT invent any facts
- Use ONLY provided signals, evidence, and meta-paths
- Mention ONLY positive evidence
- The explanation must be written as a direct message to the user
- First write in English, then in Russian
- Each language version must be 2 to 4 sentences
- English and Russian texts must be natural and user-friendly
- If interaction-related signal is present, describe it as the main reason
- Mention genres or actors only if they exist in the evidence
- Do not mention technical terms such as "edge type", "graph", "importance", "score", or "meta-path"
- The tone should sound like:
  "This movie was recommended to you because previously you..."

Structured evidence:
{payload}

Output format:

EN:
<English explanation addressed to the user>

RU:
<Russian explanation addressed to the user>
""".strip()

In [56]:
def build_positive_llm_payload(
        explanation,
        user_id: int,
        movie_idx: int,
        idx2item: dict,
        idx2feature_maps: dict,
        movie_id_to_title: dict,
        actor_id_to_name: dict,
        genre_id_to_name: dict,
        top_k_features: int = 3,
        top_k_support_movies: int = 3,
        min_positive_importance: float = 1e-4,
):
    """
    Build LLM payload from LocalExplanation:
    - only positive importance
    - readable movie / actor / genre names
    - meta-path style evidence
    """
    raw_movie_id = idx2item.get(movie_idx, movie_idx)
    readable_movie_title = movie_id_to_title.get(raw_movie_id, str(raw_movie_id))

    payload = {
        "user_id": int(user_id),
        "recommended_movie_id": int(raw_movie_id),
        "recommended_movie_title": readable_movie_title,
        "base_score": float(explanation.base_score),
        "positive_signals": [],
        "local_evidence": {},
        "meta_paths": [],
    }

    # positive signals only
    for row in explanation.edge_group_importance:
        if row.importance > min_positive_importance:
            payload["positive_signals"].append({
                "edge_type": tuple(row.edge_type),
                "importance": round(float(row.importance), 4),
            })

    # local evidence with readable names
    for feature_group_name, feature_rows in explanation.local_feature_evidence.items():
        converted_rows = []

        for row in feature_rows[:top_k_features]:
            raw_feature_id = idx2feature_maps.get(feature_group_name, {}).get(
                row.feature_node_idx,
                row.feature_node_idx,
            )

            if feature_group_name == "actor":
                readable_feature = actor_id_to_name.get(raw_feature_id, str(raw_feature_id))
            elif feature_group_name == "genre":
                readable_feature = genre_id_to_name.get(raw_feature_id, str(raw_feature_id))
            else:
                readable_feature = str(raw_feature_id)

            raw_support_movie_ids = [
                idx2item.get(int(m), int(m))
                for m in row.support_movies[:top_k_support_movies]
            ]
            readable_support_movies = [
                movie_id_to_title.get(mid, str(mid))
                for mid in raw_support_movie_ids
            ]

            converted_rows.append({
                "feature_id": int(raw_feature_id),
                "feature_name": readable_feature,
                "support_count": int(row.support_count),
                "support_movies": readable_support_movies,
            })

            for support_movie_title in readable_support_movies[:2]:
                payload["meta_paths"].append(
                    f"user → {support_movie_title} → {feature_group_name}:{readable_feature} → {readable_movie_title}"
                )

        if converted_rows:
            payload["local_evidence"][feature_group_name] = converted_rows

    return payload

In [57]:
def generate_bilingual_explanation_with_groq(
    client,
    payload: dict,
    model_name: str = GROQ_MODEL_NAME,
    temperature: float = 0.2,
) -> str:
    prompt = build_bilingual_user_prompt(payload)

    response = client.chat.completions.create(
        model=model_name,
        temperature=temperature,
        messages=[
            {
                "role": "system",
                "content": "You generate concise bilingual recommendation explanations for users."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
    )
    return response.choices[0].message.content.strip()

In [58]:
def run_llm_explanation_demo(
    explanation,
    user_id: int,
    movie_id: int,
    idx2item: dict,
    idx2feature_maps: dict,
    client: OpenAI,
    model_name: str = GROQ_MODEL_NAME,
    top_k_features: int = 3,
):
    """
    Full pipeline:
    explanation -> payload -> prompt -> LLM answer
    """
    payload = build_positive_llm_payload(
        explanation=explanation,
        user_id=user_id,
        movie_id=movie_id,
        idx2item=idx2item,
        idx2feature_maps=idx2feature_maps,
        top_k_features=top_k_features,
    )

    prompt = build_bilingual_user_prompt(payload)
    answer = generate_bilingual_explanation_with_groq(
        client=client,
        payload=payload,
        model_name=model_name,
    )

    return {
        "payload": payload,
        "prompt": prompt,
        "answer": answer,
    }

In [67]:
payload = build_positive_llm_payload(
    explanation=selected_explanation,
    user_id=example_user_id,
    movie_idx=example_movie_idx,   # internal movie index
    idx2item=cold_artifacts["idx2item"],
    idx2feature_maps=cold_artifacts["idx2feature_maps"],
    movie_id_to_title=movie_id_to_title,
    actor_id_to_name=actor_id_to_name,
    genre_id_to_name=genre_id_to_name,
    top_k_features=3,
    top_k_support_movies=3,
)

print("=== PAYLOAD ===")
print(payload)

print("\n=== PROMPT ===")
print(build_bilingual_user_prompt(payload))

# =========================
# LLM ANSWER
# =========================
print("\n=== LLM ANSWER ===")
answer = generate_bilingual_explanation_with_groq(
    client=client,
    payload=payload,
    model_name=GROQ_MODEL_NAME,
)
print(answer)

=== PAYLOAD ===
{'user_id': 888, 'recommended_movie_id': 3505, 'recommended_movie_title': 'No Way Out', 'base_score': 2.760603904724121, 'positive_signals': [{'edge_type': ('movie', 'rev_interacts', 'user'), 'importance': 12.4561}, {'edge_type': ('actor', 'rev_has_actor', 'movie'), 'importance': 6.6995}, {'edge_type': ('user', 'interacts', 'movie'), 'importance': 1.8779}, {'edge_type': ('movie', 'has_genre', 'genre'), 'importance': 0.0002}], 'local_evidence': {'actor': [{'feature_id': 59, 'feature_name': 'Gene Hackman', 'support_count': 1, 'support_movies': ['Superman']}]}, 'meta_paths': ['user → Superman → actor:Gene Hackman → No Way Out']}

=== PROMPT ===
You are explaining a movie recommendation to a user.

Use ONLY the structured evidence below.

Rules:
- Do NOT invent any facts
- Use ONLY provided signals, evidence, and meta-paths
- Mention ONLY positive evidence
- The explanation must be written as a direct message to the user
- First write in English, then in Russian
- Each lang

In [68]:
analyzer = RecommendationSubgraphAnalyzer(
    model=cold_artifacts["model"],
    data=cold_artifacts["data"],
    schema=cold_artifacts["schema"],
    train_user_items_idx=cold_artifacts["train_user_items_idx"],
    idx2item=cold_artifacts["idx2item"],
    idx2feature_maps=cold_artifacts["idx2feature_maps"],
    device=device,
)

example_user_id = None
example_user_idx = None
example_movie_id = None
example_movie_idx = None
selected_explanation = None

for user_id, recs in cold_artifacts["recommendations"].items():
    if not recs:
        continue

    recommended_movie_id = recs[0]

    gt_movies = (
        cold_artifacts["test_df_eval"]
        .loc[cold_artifacts["test_df_eval"]["user_id"] == user_id, "movie_id"]
        .drop_duplicates()
        .tolist()
    )

    is_relevant = recommended_movie_id in gt_movies
    print(f"user {user_id} -> rec {recommended_movie_id} -> relevant: {is_relevant}")

    if not is_relevant:
        continue

    user_idx = cold_artifacts["user2idx"][user_id]
    movie_idx = cold_artifacts["item2idx"][recommended_movie_id]

    explanation = analyzer.explain(
        user_idx=user_idx,
        movie_idx=movie_idx,
    )

    positive_actor = False
    positive_genre = False

    for row in explanation.edge_group_importance:
        edge_type = row.edge_type
        importance = row.importance
        rel = edge_type[1]

        if "actor" in rel and importance > 1:
            positive_actor = True

    has_positive_context = positive_actor or positive_genre

    print(
        f"   positive_actor={positive_actor}, "
        f"positive_genre={positive_genre}, "
        f"has_positive_context={has_positive_context}"
    )

    if has_positive_context:
        example_user_id = user_id
        example_user_idx = user_idx
        example_movie_id = recommended_movie_id
        example_movie_idx = movie_idx
        selected_explanation = explanation
        break

if example_user_id is None:
    raise ValueError(
        "No relevant top-1 recommendation found with positive actor or genre importance."
    )

print("\n✅ Selected relevant context-driven case")
print("Example user_id:", example_user_id)
print("Recommended movie_id:", example_movie_id)

analyzer.pretty_print(selected_explanation, top_k_features=5)

user 25 -> rec 258 -> relevant: False
user 27 -> rec 2127 -> relevant: False
user 30 -> rec 173 -> relevant: False
user 32 -> rec 3149 -> relevant: False
user 33 -> rec 1488 -> relevant: False
user 37 -> rec 173 -> relevant: False
user 41 -> rec 3699 -> relevant: False
user 48 -> rec 1894 -> relevant: False
user 65 -> rec 2395 -> relevant: False
user 66 -> rec 1488 -> relevant: False
user 68 -> rec 2040 -> relevant: False
user 80 -> rec 3149 -> relevant: False
user 86 -> rec 3519 -> relevant: False
user 88 -> rec 351 -> relevant: True
   positive_actor=True, positive_genre=False, has_positive_context=True

✅ Selected relevant context-driven case
Example user_id: 88
Recommended movie_id: 351
User idx: 87
Movie idx: 326 -> raw movie id: 351
Base score: 1.5681

[Edge-group importance]
('movie', 'rev_interacts', 'user'): Δscore = 6.5963 (masked=-5.0281)
('user', 'interacts', 'movie'): Δscore = 5.2217 (masked=-3.6535)
('actor', 'rev_has_actor', 'movie'): Δscore = 3.8981 (masked=-2.3300)
('g

In [69]:
payload = build_positive_llm_payload(
    explanation=selected_explanation,
    user_id=example_user_id,
    movie_idx=example_movie_idx,   # internal movie index
    idx2item=cold_artifacts["idx2item"],
    idx2feature_maps=cold_artifacts["idx2feature_maps"],
    movie_id_to_title=movie_id_to_title,
    actor_id_to_name=actor_id_to_name,
    genre_id_to_name=genre_id_to_name,
    top_k_features=3,
    top_k_support_movies=3,
)

print("=== PAYLOAD ===")
print(payload)

print("\n=== PROMPT ===")
print(build_bilingual_user_prompt(payload))

# =========================
# LLM ANSWER
# =========================
print("\n=== LLM ANSWER ===")
answer = generate_bilingual_explanation_with_groq(
    client=client,
    payload=payload,
    model_name=GROQ_MODEL_NAME,
)
print(answer)

=== PAYLOAD ===
{'user_id': 88, 'recommended_movie_id': 351, 'recommended_movie_title': 'Corrina, Corrina', 'base_score': 1.5681390762329102, 'positive_signals': [{'edge_type': ('movie', 'rev_interacts', 'user'), 'importance': 6.5963}, {'edge_type': ('user', 'interacts', 'movie'), 'importance': 5.2217}, {'edge_type': ('actor', 'rev_has_actor', 'movie'), 'importance': 3.8981}, {'edge_type': ('genre', 'rev_has_genre', 'movie'), 'importance': 3.0936}, {'edge_type': ('movie', 'has_actor', 'actor'), 'importance': 0.5778}, {'edge_type': ('movie', 'has_genre', 'genre'), 'importance': 0.0004}], 'local_evidence': {'genre': [{'feature_id': 8, 'feature_name': 'Drama', 'support_count': 1, 'support_movies': ['The Color Purple']}], 'actor': [{'feature_id': 458, 'feature_name': 'Whoopi Goldberg', 'support_count': 1, 'support_movies': ['The Color Purple']}]}, 'meta_paths': ['user → The Color Purple → genre:Drama → Corrina, Corrina', 'user → The Color Purple → actor:Whoopi Goldberg → Corrina, Corrina']